# Pegasus-X Base: Train and Generate Meta-Reviews
This notebook trains Pegasus-X Base on ICLR meta-review data and generates predictions.

In [ ]:
!pip install -q transformers datasets accelerate

In [ ]:
import tempfile
import json
import torch
import os
from datasets import load_dataset
from transformers import (
    PegasusTokenizer,
    PegasusXForConditionalGeneration,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
)
from transformers.trainer_utils import get_last_checkpoint

In [ ]:
MODEL_NAME = 'google/pegasus-x-base'
TRAIN_FILE = '/content/data/train.jsonl'
VAL_FILE   = '/content/data/val.jsonl'
TEST_FILE  = '/content/data/test.jsonl'
OUTPUT_DIR = 'pegasus_x_meta_review'
MAX_INPUT = 1024
MAX_TARGET = 384
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
def filter_jsonl_file(input_filepath):
    temp_file = tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.jsonl')
    malformed = 0
    with open(input_filepath, 'r') as infile:
        for line in infile:
            try:
                obj = json.loads(line)
                temp_file.write(json.dumps(obj)+'\n')
            except:
                malformed += 1
    temp_file.close()
    print(f'Skipped {malformed} malformed lines')
    return temp_file.name

TRAIN_FILE = filter_jsonl_file(TRAIN_FILE)
VAL_FILE   = filter_jsonl_file(VAL_FILE)
TEST_FILE  = filter_jsonl_file(TEST_FILE)

In [ ]:
dataset = load_dataset(
    'json',
    data_files={'train': TRAIN_FILE, 'validation': VAL_FILE, 'test': TEST_FILE}
)
print(dataset)

In [ ]:
tokenizer = PegasusTokenizer.from_pretrained(MODEL_NAME)
model = PegasusXForConditionalGeneration.from_pretrained(MODEL_NAME)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

In [ ]:
def preprocess(example):
    inputs = tokenizer(
        str(example.get('input_text','')),
        max_length=MAX_INPUT,
        truncation=True,
        padding='max_length'
    )
    labels = tokenizer(
        text_target=str(example.get('target_text','')),
        max_length=MAX_TARGET,
        truncation=True,
        padding='max_length'
    )
    inputs['labels'] = labels['input_ids']
    return inputs

tokenized_ds = dataset.map(preprocess, remove_columns=dataset['train'].column_names)

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=3e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    evaluation_strategy='steps',
    save_strategy='steps',
    eval_steps=1000,
    save_steps=1000,
    logging_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

last_checkpoint = get_last_checkpoint(OUTPUT_DIR)
if last_checkpoint:
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Training complete')

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm
import csv

tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(OUTPUT_DIR).to(device)
model.eval()

def extract_decision(text):
    if 'DECISION:' in text:
        first = text.split('DECISION:')[1].split('\n')[0].upper()
        if first.startswith('ACCEPT'): return 'ACCEPT'
        if first.startswith('REJECT'): return 'REJECT'
    return 'UNKNOWN'

rows = []
with open(TEST_FILE,'r') as f:
    for line in tqdm(f):
        ex=json.loads(line)
        inputs=tokenizer(ex['input_text'],return_tensors='pt',truncation=True,max_length=1024).to(device)
        with torch.no_grad():
            out=model.generate(**inputs,max_new_tokens=300,num_beams=4)
        pred=tokenizer.decode(out[0],skip_special_tokens=True)
        rows.append({
            'paper_id':ex['paper_id'],
            'true_decision':extract_decision(ex['target_text']),
            'pred_decision':extract_decision(pred),
            'pred_meta_review':pred
        })

os.makedirs('data/predictions',exist_ok=True)
with open('data/predictions/predictions.csv','w',newline='',encoding='utf-8') as f:
    writer=csv.DictWriter(f,fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)
print('Predictions saved')